<a href="https://colab.research.google.com/github/sharniks/langchain-fundamentals/blob/main/tools_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.8 MB/s eta 0:00:00


# Built-in Tool - DuckDuckGo Search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('fifa world cup 2026 news')

print(results)

8 hours ago - Arsene Wenger says he accepts the hydration breaks introduced at the 2026 World Cup have not been popular and Fifa will review their impact after the tournament. 12 hours ago - The latest in the FIFA 'Letters that Unite' series features Guillermo Ochoa, veteran goalkeeper of World Cup 2026 co-hosts Mexico, and his daughter Lucciana.The latest in the FIFA 'Letters that Unite' series features Guillermo Ochoa, veteran ... 6 hours ago - France v Spain | Semi-final | FIFA World Cup 2026™ | HighlightsFrance v Spain | Semi-final | FIFA World Cup 2026™ | Highlights ... Argentina v Switzerland | Quarter-final | FIFA World Cup 2026™ | HighlightsArgentina v Switzerland | Quarter-final | FIFA World Cup 2026™ | Highlights We're down to the final of the 2026 FIFA World Cup, as there are now just two teams still harboring dreams of becoming world champions. ... Messi first met Yamal as a baby. Their next meeting will be in a World Cup final ... They’re playing for rings as well. FIFA Pr

In [ ]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# Built-in Tool - Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
sample_data



# Custom Tools

In [ ]:
from langchain_core.tools import tool

In [ ]:
# Step 1 - create a function
def multiply(a,b):
  """Multiply two numbers"""
  return a*b

In [ ]:
# Step 2 - add type hints
def multiply(a:int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [ ]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [ ]:
result = multiply.invoke({"a":3, "b":5})

In [ ]:
print(result)

15


In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [ ]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


# Method 2 - Using StructuredTool

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [ ]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="First number")
  b: int = Field(required=True, description="Second number")

/tmp/ipykernel_1211/455087635.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="First number")
/tmp/ipykernel_1211/455087635.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="Second number")


In [ ]:
def multiply_func(a: int, b:int) -> int:
  return a * b

In [ ]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [ ]:
result = multiply_tool.invoke({'a':3,'b':3})

In [ ]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'First number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'integer'}}


# Method 3 - Using BaseTool Class

In [ ]:
from langchain.tools import BaseTool
from typing import Type

In [ ]:
# arg schema using pydantic
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="First number")
  b: int = Field(required=True, description="Second number")

/tmp/ipykernel_1211/3618704352.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="First number")
/tmp/ipykernel_1211/3618704352.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="Second number")


In [ ]:
class MultiplyTool(BaseTool):
  name: str = "multiply"
  description: str = "Multiply Two Numbers"

  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a: int, b: int) -> int:
    return a * b

In [ ]:
multiply_tool = MultiplyTool()

In [ ]:
result = multiply_tool.invoke({'a':3,'b':3})

In [ ]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply Two Numbers
{'a': {'description': 'First number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'integer'}}


# Toolkit

In [ ]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b:int) -> int:
  """Add two numbers"""
  return a+b

@tool
def multiply(a: int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [ ]:
class MathToolkit:
  def get_tools(self):
    return[add, multiply]

In [ ]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
  print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
